# Commentary Evaluation Pipeline (LLM-as-a-Judge)

**Two stages:**
1. **Generation** — Generate NL move commentary (with or without engine info)
2. **Evaluation** — Tool-augmented atom-level judge:
   - **Decompose** candidate into atomic claims
   - **Verify** each atom against the board (tool-augmented)
   - **Match** candidate atoms to gold atoms
   - **Quality check** — does the explanation correctly assess move quality?

**Inputs:** `data/review_accepted.jsonl` (gold atoms), `data/review_demo.jsonl` (few-shot demos)

**Metrics:** Factual Precision, Recall (gold coverage), Quality correctness

---
## Setup & Data Loading

In [ ]:
import json
import os
import math
import random
import numpy as np
import chess
import chess.engine
import chess.svg
from IPython.display import display, SVG, Markdown, HTML
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv()

STOCKFISH_PATH = '/opt/homebrew/bin/stockfish'
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('')), 'evaluation', 'data')

K = 0.00368208


def cp_to_winpct(cp):
    """Lichess win% formula."""
    return 50 + 50 * math.tanh(K * cp / 2)


def get_engine_analysis(fen, move_uci=None, depth=22, multipv=3):
    """Get Stockfish top lines. If move_uci given and not in top-N, fetch separately."""
    board = chess.Board(fen)
    with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
        result = engine.analyse(board, chess.engine.Limit(depth=depth), multipv=multipv)
        lines = []
        for info in result:
            score = info['score'].white()
            pv = info.get('pv', [])
            if pv:
                b = board.copy()
                san_moves = []
                for m in pv[:8]:
                    san_moves.append(b.san(m))
                    b.push(m)
                cp = score.score(mate_score=10000)
                mate = score.mate()
                eval_str = f'M{mate}' if mate is not None else f'{cp/100:+.2f}'
                lines.append({
                    'move_uci': pv[0].uci(), 'move_san': board.san(pv[0]),
                    'eval': eval_str, 'pv_san': ' '.join(san_moves),
                    'cp': cp, 'mate': mate, 'is_top': True,
                })

        if move_uci is not None:
            if not any(l['move_uci'] == move_uci for l in lines):
                move_obj = chess.Move.from_uci(move_uci)
                move_san = board.san(move_obj)
                b2 = board.copy()
                b2.push(move_obj)
                info2 = engine.analyse(b2, chess.engine.Limit(depth=depth))
                sc2 = info2['score'].white()
                cp2 = sc2.score(mate_score=10000)
                mate2 = sc2.mate()
                eval2 = f'M{mate2}' if mate2 is not None else f'{cp2/100:+.2f}'
                pv2 = info2.get('pv', [])
                pv_sans = [move_san]
                b3 = b2.copy()
                for m in pv2[:7]:
                    pv_sans.append(b3.san(m))
                    b3.push(m)
                lines.append({
                    'move_uci': move_uci, 'move_san': move_san,
                    'eval': eval2, 'pv_san': ' '.join(pv_sans),
                    'cp': cp2, 'mate': mate2, 'is_top': False,
                })

    return lines


print(f'Data dir: {DATA_DIR}')

In [ ]:
ACCEPTED_PATH = os.path.join(DATA_DIR, 'review_accepted.jsonl')
DEMO_PATH = os.path.join(DATA_DIR, 'review_demo.jsonl')

with open(ACCEPTED_PATH) as f:
    accepted_positions = [json.loads(line) for line in f]

with open(DEMO_PATH) as f:
    review_demos = [json.loads(line) for line in f]

# Raw entries for find_entry()
with open(os.path.join(DATA_DIR, 'logical_chess_optimal_move.jsonl')) as f:
    optimal_moves = [json.loads(line) for line in f]
with open(os.path.join(DATA_DIR, 'logical_chess_other_move.jsonl')) as f:
    other_moves = [json.loads(line) for line in f]
_all_raw = optimal_moves + other_moves

def find_entry(fen, move_uci):
    for e in _all_raw:
        if e['fen'] == fen and e['move_uci'] == move_uci:
            return e
    raise ValueError(f'Not found: {fen} {move_uci}')

print(f'Accepted positions: {len(accepted_positions)}')
print(f'Demo positions: {len(review_demos)}')
print(f'Raw entries: {len(_all_raw)}')

---
## Chess Tools & API Abstraction

In [ ]:
import json as _json

def tool_get_legal_moves(fen):
    board = chess.Board(fen)
    moves = [board.san(m) for m in board.legal_moves]
    return _json.dumps({"moves": moves, "count": len(moves)})

def tool_get_piece_at(fen, square):
    board = chess.Board(fen)
    sq = chess.parse_square(square)
    p = board.piece_at(sq)
    if p is None:
        return _json.dumps({"square": square, "piece": "empty"})
    color = "white" if p.color == chess.WHITE else "black"
    return _json.dumps({"square": square, "piece": f"{color} {chess.piece_name(p.piece_type)}"})

def tool_get_attacks(fen, square):
    board = chess.Board(fen)
    sq = chess.parse_square(square)
    p = board.piece_at(sq)
    if p is None:
        return _json.dumps({"error": f"no piece on {square}"})
    attacks = sorted([chess.square_name(a) for a in board.attacks(sq)])
    return _json.dumps({"piece": p.symbol(), "square": square, "attacks": attacks})

def tool_get_attackers(fen, square, color):
    board = chess.Board(fen)
    sq = chess.parse_square(square)
    c = chess.WHITE if color == "white" else chess.BLACK
    attackers = [[chess.piece_name(board.piece_at(a).piece_type), chess.square_name(a)]
                 for a in board.attackers(c, sq)]
    return _json.dumps({"square": square, "color": color, "attackers": attackers})

def tool_is_pinned(fen, square):
    board = chess.Board(fen)
    sq = chess.parse_square(square)
    p = board.piece_at(sq)
    if p is None:
        return _json.dumps({"error": f"no piece on {square}"})
    pinned = board.is_pinned(p.color, sq)
    result = {"square": square, "pinned": pinned}
    if pinned:
        result["pin_ray"] = str(chess.SquareSet(board.pin(p.color, sq)))
    return _json.dumps(result)

def tool_is_check(fen):
    board = chess.Board(fen)
    in_check = board.is_check()
    result = {"in_check": in_check}
    if in_check:
        result["checkers"] = [chess.square_name(sq) for sq in board.checkers()]
    return _json.dumps(result)

def tool_try_variation(fen, moves):
    board = chess.Board(fen)
    played = []
    for san in moves:
        try:
            move = board.parse_san(san)
            played.append(board.san(move))
            board.push(move)
        except Exception as e:
            return _json.dumps({"legal": False, "error": f"{san}: {str(e)}", "played": played})
    result = {"legal": True, "played": played, "resulting_fen": board.fen()}
    if board.is_checkmate():
        result["eval"] = "checkmate"
    elif board.is_stalemate():
        result["eval"] = "stalemate"
    else:
        with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
            info = engine.analyse(board, chess.engine.Limit(depth=18))
            sc = info['score'].white()
            mate = sc.mate()
            result["eval"] = f"M{mate}" if mate is not None else f"{sc.score(mate_score=10000)/100:+.2f}"
    return _json.dumps(result)

def tool_get_engine_eval(fen, depth=20):
    board = chess.Board(fen)
    with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
        info = engine.analyse(board, chess.engine.Limit(depth=int(depth)))
        sc = info['score'].white()
        pv = info.get('pv', [])
        mate = sc.mate()
        ev = f"M{mate}" if mate is not None else f"{sc.score(mate_score=10000)/100:+.2f}"
        best = board.san(pv[0]) if pv else "?"
    return _json.dumps({"eval": ev, "best_move": best})

def tool_get_top_moves(fen, n=3, depth=20):
    """Get top N engine moves with evals and PV lines."""
    board = chess.Board(fen)
    with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
        result = engine.analyse(board, chess.engine.Limit(depth=int(depth)), multipv=int(n))
        moves = []
        for info in result:
            score = info['score'].white()
            pv = info.get('pv', [])
            if pv:
                b = board.copy()
                san_moves = []
                for m in pv[:6]:
                    san_moves.append(b.san(m))
                    b.push(m)
                cp = score.score(mate_score=10000)
                mate = score.mate()
                ev = f"M{mate}" if mate is not None else f"{cp/100:+.2f}"
                moves.append({"move": board.san(pv[0]), "eval": ev, "pv": " ".join(san_moves)})
    return _json.dumps({"top_moves": moves})

def tool_eval_move(fen, move, depth=20):
    """Evaluate a specific move: eval, best move, wp_loss, quality."""
    board = chess.Board(fen)
    def wp(cp):
        return 50 + 50 * math.tanh(K * cp / 2)
    with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
        root_info = engine.analyse(board, chess.engine.Limit(depth=int(depth)))
        root_cp = root_info['score'].white().score(mate_score=10000)
        root_pv = root_info.get('pv', [])
        best_move_san = board.san(root_pv[0]) if root_pv else "?"

        m = board.parse_san(move)
        move_san = board.san(m)
        b2 = board.copy()
        b2.push(m)
        move_info = engine.analyse(b2, chess.engine.Limit(depth=int(depth)))
        move_cp = move_info['score'].white().score(mate_score=10000)
        move_mate = move_info['score'].white().mate()
        move_eval_str = f"M{move_mate}" if move_mate is not None else f"{move_cp/100:+.2f}"

    if board.turn == chess.WHITE:
        wp_loss_val = wp(root_cp) - wp(move_cp)
    else:
        wp_loss_val = wp(move_cp) - wp(root_cp)
    wp_loss_val = round(max(0, wp_loss_val), 1)
    quality = "good" if wp_loss_val < 5 else ("bad" if wp_loss_val < 20 else "blunder")

    return _json.dumps({
        "move": move_san, "move_eval": move_eval_str,
        "best_move": best_move_san, "wp_loss": wp_loss_val, "quality": quality,
    })

def tool_compare_moves(fen, move_a, move_b, depth=20):
    board = chess.Board(fen)
    def wp(cp):
        return 50 + 50 * math.tanh(K * cp / 2)
    with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
        ba = board.copy(); ba.push(board.parse_san(move_a))
        ia = engine.analyse(ba, chess.engine.Limit(depth=int(depth)))
        sa = ia['score'].white().score(mate_score=10000)
        bb = board.copy(); bb.push(board.parse_san(move_b))
        ib = engine.analyse(bb, chess.engine.Limit(depth=int(depth)))
        sb = ib['score'].white().score(mate_score=10000)
    return _json.dumps({
        "move_a": move_a, "eval_a": f"{sa/100:+.2f}",
        "move_b": move_b, "eval_b": f"{sb/100:+.2f}",
        "wp_diff": round(abs(wp(sa) - wp(sb)), 1)
    })

def tool_make_move(fen, move):
    board = chess.Board(fen)
    m = board.parse_san(move)
    gives_check = board.gives_check(m)
    board.push(m)
    return _json.dumps({"fen": board.fen(), "gives_check": gives_check})

_PIECE_NAMES = {"pawn": chess.PAWN, "knight": chess.KNIGHT, "bishop": chess.BISHOP,
                "rook": chess.ROOK, "queen": chess.QUEEN, "king": chess.KING}

def tool_get_squares(fen, piece, color=None):
    """Find all squares of a piece type, optionally by color."""
    board = chess.Board(fen)
    pt = _PIECE_NAMES.get(piece.lower())
    if pt is None:
        return _json.dumps({"error": f"unknown piece: {piece}"})
    colors = []
    if color is None:
        colors = [chess.WHITE, chess.BLACK]
    elif color == "white":
        colors = [chess.WHITE]
    elif color == "black":
        colors = [chess.BLACK]
    else:
        return _json.dumps({"error": f"unknown color: {color}"})
    results = []
    for c in colors:
        cname = "white" if c == chess.WHITE else "black"
        for sq in board.pieces(pt, c):
            results.append({"square": chess.square_name(sq), "color": cname})
    return _json.dumps({"piece": piece, "squares": results})

def tool_get_material(fen):
    """Get material count for both sides."""
    board = chess.Board(fen)
    result = {}
    for color_name, color in [("white", chess.WHITE), ("black", chess.BLACK)]:
        pieces = {}
        for pt, name in [(chess.PAWN, "pawns"), (chess.KNIGHT, "knights"),
                         (chess.BISHOP, "bishops"), (chess.ROOK, "rooks"),
                         (chess.QUEEN, "queens")]:
            count = len(board.pieces(pt, color))
            if count > 0:
                pieces[name] = count
        result[color_name] = pieces
    return _json.dumps(result)


TOOL_FUNCTIONS = {
    "get_legal_moves": tool_get_legal_moves, "get_piece_at": tool_get_piece_at,
    "get_attacks": tool_get_attacks, "get_attackers": tool_get_attackers,
    "is_pinned": tool_is_pinned, "is_check": tool_is_check,
    "try_variation": tool_try_variation, "get_engine_eval": tool_get_engine_eval,
    "get_top_moves": tool_get_top_moves, "eval_move": tool_eval_move,
    "compare_moves": tool_compare_moves, "make_move": tool_make_move,
    "get_squares": tool_get_squares, "get_material": tool_get_material,
}

def _schema(name, desc, props, required):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": required}}}

_fen_prop = {"fen": {"type": "string", "description": "FEN of the position"}}
_sq_prop = {"square": {"type": "string", "description": "Square name, e.g. 'd3'"}}

TOOL_SCHEMAS_OPENAI = [
    _schema("get_legal_moves", "Get all legal moves in SAN.", _fen_prop, ["fen"]),
    _schema("get_piece_at", "Get the piece on a square.", {**_fen_prop, **_sq_prop}, ["fen", "square"]),
    _schema("get_attacks", "Get squares attacked by piece on a square.", {**_fen_prop, **_sq_prop}, ["fen", "square"]),
    _schema("get_attackers", "Get pieces of a color attacking a square.",
            {**_fen_prop, **_sq_prop, "color": {"type": "string", "enum": ["white", "black"]}},
            ["fen", "square", "color"]),
    _schema("is_pinned", "Check if a piece is pinned.", {**_fen_prop, **_sq_prop}, ["fen", "square"]),
    _schema("is_check", "Check if side to move is in check.", _fen_prop, ["fen"]),
    _schema("try_variation", "Try a sequence of SAN moves. Returns resulting FEN.",
            {**_fen_prop, "moves": {"type": "array", "items": {"type": "string"}}}, ["fen", "moves"]),
    _schema("get_engine_eval", "Get Stockfish eval + best move.",
            {**_fen_prop, "depth": {"type": "integer"}}, ["fen"]),
    _schema("get_top_moves", "Get top N engine moves with evals and PV lines.",
            {**_fen_prop, "n": {"type": "integer"}, "depth": {"type": "integer"}}, ["fen"]),
    _schema("eval_move", "Evaluate a specific move: eval, best move, wp_loss, quality.",
            {**_fen_prop, "move": {"type": "string"}, "depth": {"type": "integer"}}, ["fen", "move"]),
    _schema("compare_moves", "Compare eval of two moves.",
            {**_fen_prop, "move_a": {"type": "string"}, "move_b": {"type": "string"},
             "depth": {"type": "integer"}}, ["fen", "move_a", "move_b"]),
    _schema("make_move", "Make a move, return new FEN.",
            {**_fen_prop, "move": {"type": "string"}}, ["fen", "move"]),
    _schema("get_squares", "Find all squares of a piece type, optionally by color.",
            {**_fen_prop, "piece": {"type": "string", "enum": ["pawn","knight","bishop","rook","queen","king"]},
             "color": {"type": "string", "enum": ["white", "black"]}}, ["fen", "piece"]),
    _schema("get_material", "Get material count for both sides.", _fen_prop, ["fen"]),
]

TOOL_SCHEMAS_ANTHROPIC = [
    {"name": s["function"]["name"], "description": s["function"]["description"],
     "input_schema": s["function"]["parameters"]} for s in TOOL_SCHEMAS_OPENAI
]

def execute_tool(name, arguments):
    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return _json.dumps({"error": f"unknown tool: {name}"})
    try:
        return fn(**arguments)
    except Exception as e:
        return _json.dumps({"error": str(e)})

print(f'Tools: {list(TOOL_FUNCTIONS.keys())}')

In [ ]:
# ── API Abstraction ───────────────────────────────────────────────────────

@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict

@dataclass
class LLMResponse:
    text: str
    tool_calls: list


def call_openai(messages, tools=None, model="gpt-4o", temperature=0.3):
    import openai
    client = openai.OpenAI()
    token_key = "max_completion_tokens" if any(m in model for m in ("gpt-4.1", "gpt-5", "o3", "o4")) else "max_tokens"
    kwargs = dict(model=model, messages=messages, temperature=temperature, **{token_key: 2048})
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    resp = client.chat.completions.create(**kwargs)
    msg = resp.choices[0].message
    text = msg.content or ""
    tc = []
    if msg.tool_calls:
        for c in msg.tool_calls:
            tc.append(ToolCall(id=c.id, name=c.function.name,
                               arguments=_json.loads(c.function.arguments)))
    return LLMResponse(text=text, tool_calls=tc), msg


def call_anthropic(messages, tools=None, model="claude-sonnet-4-5-20250929",
                   system=None, temperature=0.3):
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=model, max_tokens=2048, temperature=temperature)
    if system:
        kwargs["system"] = system
    api_messages = [m for m in messages if m["role"] != "system"]
    kwargs["messages"] = api_messages
    if tools:
        kwargs["tools"] = tools
    resp = client.messages.create(**kwargs)
    text = ""
    tc = []
    for block in resp.content:
        if block.type == "text":
            text += block.text
        elif block.type == "tool_use":
            tc.append(ToolCall(id=block.id, name=block.name, arguments=block.input))
    return LLMResponse(text=text, tool_calls=tc), resp


def call_llm(messages, tools=None, provider="openai", model=None, temperature=0.3):
    if provider == "openai":
        return call_openai(messages, tools=tools, model=model or "gpt-4o", temperature=temperature)
    elif provider == "anthropic":
        m = model or "claude-sonnet-4-5-20250929"
        sys_msg = next((msg["content"] for msg in messages if msg["role"] == "system"), None)
        return call_anthropic(messages, tools=tools, model=m, system=sys_msg, temperature=temperature)
    else:
        raise ValueError(f"Unknown provider: {provider}")


PROVIDER = "openai"
MODEL = "gpt-4o"
MAX_TOOL_ROUNDS = 10


def _call_with_tools(messages, provider=PROVIDER, model=MODEL, temperature=0.3,
                     max_rounds=MAX_TOOL_ROUNDS, verbose=True):
    """Run LLM with tool access, handling multi-round tool calls.
    Returns (final_text, tool_log)."""
    tool_schemas = TOOL_SCHEMAS_OPENAI if provider == "openai" else TOOL_SCHEMAS_ANTHROPIC
    tool_log = []

    for _ in range(max_rounds):
        resp, raw = call_llm(messages, tools=tool_schemas, provider=provider,
                             model=model, temperature=temperature)

        if not resp.tool_calls:
            return resp.text.strip() if resp.text else "", tool_log

        if provider == "openai":
            messages.append({
                "role": "assistant", "content": resp.text or None,
                "tool_calls": [{"id": tc.id, "type": "function",
                                "function": {"name": tc.name,
                                             "arguments": _json.dumps(tc.arguments)}}
                               for tc in resp.tool_calls],
            })
            for tc in resp.tool_calls:
                result = execute_tool(tc.name, tc.arguments)
                tool_log.append({"tool": tc.name, "args": tc.arguments, "result": result})
                if verbose:
                    print(f"    tool: {tc.name}({tc.arguments}) -> {result[:80]}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        elif provider == "anthropic":
            content_blocks = []
            if resp.text:
                content_blocks.append({"type": "text", "text": resp.text})
            for tc in resp.tool_calls:
                content_blocks.append({"type": "tool_use", "id": tc.id,
                                       "name": tc.name, "input": tc.arguments})
            messages.append({"role": "assistant", "content": content_blocks})
            tool_results = []
            for tc in resp.tool_calls:
                result = execute_tool(tc.name, tc.arguments)
                tool_log.append({"tool": tc.name, "args": tc.arguments, "result": result})
                if verbose:
                    print(f"    tool: {tc.name}({tc.arguments}) -> {result[:80]}")
                tool_results.append({"type": "tool_result", "tool_use_id": tc.id,
                                     "content": result})
            messages.append({"role": "user", "content": tool_results})

    return resp.text.strip() if resp.text else "[max tool rounds]", tool_log


print(f'API ready. Provider: {PROVIDER}, Model: {MODEL}')

---
## Generation Template

In [ ]:
GEN_SYSTEM_PROMPT = """\
You are a chess instructor explaining moves to an intermediate player.

Given a position (FEN), the move played, and engine analysis, write a concise, \
instructive explanation of the move.

GUIDELINES:
- Explain the strategic or tactical purpose of the move.
- If the move is bad or a blunder: explain why it fails and what was better.
- Be specific: name squares, pieces, diagonals, and key variations.
- No generic chess philosophy or filler. Every sentence must be position-specific.
- Keep it concise. A simple forced move needs one sentence. \
A rich tactical position deserves a short paragraph.
- Output plain text only. No JSON, no bullet points, no headers."""

GEN_RAW_SYSTEM_PROMPT = """\
You are a chess instructor explaining moves to an intermediate player.

Given a position (FEN) and the move played, write a concise, instructive \
explanation of the move. You have access to chess analysis tools \u2014 use them \
to understand the position before writing your explanation.

GUIDELINES:
- Explain the strategic or tactical purpose of the move.
- If the move is bad or a blunder: explain why it fails and what was better.
- Be specific: name squares, pieces, diagonals, and key variations.
- No generic chess philosophy or filler. Every sentence must be position-specific.
- Keep it concise. A simple forced move needs one sentence. \
A rich tactical position deserves a short paragraph.
- Output plain text only. No JSON, no bullet points, no headers."""


def fen_to_ascii(fen):
    return str(chess.Board(fen))


def build_gen_user_prompt(entry, engine_lines, ascii=False):
    """Build user prompt: FEN + move + engine (NO commentary)."""
    board = chess.Board(entry['fen'])
    move_san = board.san(chess.Move.from_uci(entry['move_uci']))

    top_lines = [l for l in engine_lines if l.get('is_top', True)]
    played_line = next((l for l in engine_lines if not l.get('is_top', True)), None)

    pv_text = []
    for i, line in enumerate(top_lines, 1):
        pv_text.append(f"  {i}. {line['move_san']} ({line['eval']}): {line['pv_san']}")

    wp_loss = entry.get('wp_loss', 0)
    if wp_loss > 20:   quality_hint = f" [BLUNDER \u2014 wp_loss: {wp_loss:.1f}%]"
    elif wp_loss > 5:  quality_hint = f" [SUBOPTIMAL \u2014 wp_loss: {wp_loss:.1f}%]"
    else:              quality_hint = f" [wp_loss: {wp_loss:.1f}%]"

    prompt = f"FEN: {entry['fen']}\n"
    if ascii:
        prompt += f"Board:\n{fen_to_ascii(entry['fen'])}\n"
    prompt += f"Move played: {move_san}{quality_hint}\nEngine top-3:\n" + "\n".join(pv_text) + "\n"
    if played_line:
        prompt += f"Played move eval:\n  {played_line['move_san']} ({played_line['eval']}): {played_line['pv_san']}\n"
    return prompt


def build_gen_user_prompt_raw(entry, ascii=False):
    """Build user prompt: FEN + move only (no engine info)."""
    board = chess.Board(entry['fen'])
    move_san = board.san(chess.Move.from_uci(entry['move_uci']))
    prompt = f"FEN: {entry['fen']}\n"
    if ascii:
        prompt += f"Board:\n{fen_to_ascii(entry['fen'])}\n"
    prompt += f"Move played: {move_san}\n"
    return prompt


# Build demo messages
print("Building generation demos...")

GEN_DEMOS = []
GEN_RAW_DEMOS = []

for demo in review_demos:
    entry = {'fen': demo['fen'], 'move_uci': demo['move_uci'],
             'wp_loss': demo.get('wp_loss', 0)}
    el = demo.get('engine_lines') or get_engine_analysis(demo['fen'], demo['move_uci'])
    user_prompt = build_gen_user_prompt(entry, el)
    GEN_DEMOS.append({"role": "user", "content": user_prompt})
    GEN_DEMOS.append({"role": "assistant", "content": demo['demo_output']})

    raw_prompt = build_gen_user_prompt_raw(entry)
    GEN_RAW_DEMOS.append({"role": "user", "content": raw_prompt})
    GEN_RAW_DEMOS.append({"role": "assistant", "content": demo['demo_output']})

    board = chess.Board(demo['fen'])
    san = board.san(chess.Move.from_uci(demo['move_uci']))
    print(f"  {san} ({demo.get('quality', '?')}): done")

print(f"\nGeneration template ready: {len(GEN_DEMOS)//2} demos")

In [ ]:
def generate_commentary(entry, engine_lines, provider=PROVIDER, model=MODEL, ascii=False):
    """Generate NL move commentary with engine info. Returns (text, tool_log)."""
    messages = [{"role": "system", "content": GEN_SYSTEM_PROMPT}]
    messages.extend(GEN_DEMOS)
    messages.append({"role": "user", "content": build_gen_user_prompt(entry, engine_lines, ascii=ascii)})
    return _call_with_tools(messages, provider=provider, model=model, temperature=0.3)


def generate_commentary_raw(entry, provider=PROVIDER, model=MODEL, ascii=False):
    """Generate commentary from FEN + move only (tool-augmented). Returns (text, tool_log)."""
    messages = [{"role": "system", "content": GEN_RAW_SYSTEM_PROMPT}]
    messages.extend(GEN_RAW_DEMOS)
    messages.append({"role": "user", "content": build_gen_user_prompt_raw(entry, ascii=ascii)})
    return _call_with_tools(messages, provider=provider, model=model, temperature=0.3)


print('Generation functions ready.')

---
## LLM-as-Judge

Tool-augmented atom-level evaluation:
1. **Decompose** candidate explanation into atomic claims
2. **Verify** each atom against the board (tool-augmented)
3. **Match** candidate atoms to gold atoms
4. **Quality check** — does the explanation correctly assess move quality?

In [ ]:
def _parse_judge_json(text):
    """Parse JSON from judge LLM output, handling markdown fences."""
    text = text.strip()
    if text.startswith('```'):
        text = text.split('\n', 1)[1]
        if text.endswith('```'):
            text = text[:-3]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print(f"  [JSON parse error] {text[:200]}...")
        return {}


# ── Step 1: Decompose ─────────────────────────────────────────────────────

DECOMPOSE_SYSTEM = """\
Extract atomic factual claims from a chess move explanation.

Each atom = ONE verifiable claim about the position, a piece, a square, \
a variation, or an evaluation.

RULES:
1. Be specific \u2014 include piece names, squares, and concrete assertions.
2. Group logically connected setup+consequence as one atom.
3. Each atom must be self-contained with full positional context. If an atom \
references a position after a sequence of moves, include the FULL preceding \
move sequence so it can be verified independently.
4. Ignore stylistic flourishes that make no factual claim.
5. Ignore generic chess philosophy not applied to this position.

Output valid JSON: {"atoms": ["claim 1", "claim 2", ...]}"""


def decompose_to_atoms(text, provider=PROVIDER, model=MODEL):
    """Decompose NL explanation into atomic factual claims."""
    messages = [
        {"role": "system", "content": DECOMPOSE_SYSTEM},
        {"role": "user", "content": text},
    ]
    resp, _ = call_llm(messages, provider=provider, model=model, temperature=0.0)
    data = _parse_judge_json(resp.text)
    return data.get('atoms', [])


# ── Step 2: Verify ────────────────────────────────────────────────────────

VERIFY_SYSTEM = """\
You are a chess position fact-checker with access to analysis tools.

CONTEXT: You are given a PRE-MOVE position (FEN), the move played, and \
the POST-MOVE position (FEN). The explanation describes what this move does.

WHICH FEN TO USE:
- Claims about piece placement or attacks AFTER the move -> use post-move FEN.
- Claims about alternatives or what existed before the move -> use pre-move FEN.
- Variations starting from the post-move position -> use post-move FEN.

CRITICAL RULES:
1. NEVER conclude true/false based on reasoning alone. Every verdict MUST be \
supported by at least one tool call.
2. Each atom may contain MULTIPLE factual sub-claims. Verify ALL of them.
3. Use MULTIPLE tool calls when needed.
4. For forward-looking claims: verify preconditions exist.
5. For \"controls a file\" or \"dominates\": check what BOTH sides have.
6. For \"removes a defender\": verify what was actually defended BEFORE the move.
7. For move sequences: use try_variation with the FULL sequence.
8. For claims about quality: use eval_move to check actual wp_loss.

Output JSON:
{"results": [{"atom": "...", "verified": true, "reasoning": "tool showed X"}, ...], \
"assessment": "good"|"bad"|"inconclusive"}"""


def verify_atoms(atoms, fen, move_san, move_uci, provider=PROVIDER, model=MODEL):
    """Verify atoms against position using chess tools.
    Returns (results_list, assessment, tool_log)."""
    if not atoms:
        return [], 'inconclusive', []

    board = chess.Board(fen)
    board.push(chess.Move.from_uci(move_uci))
    post_fen = board.fen()

    user_msg = (
        f"Pre-move FEN: {fen}\n"
        f"Move played: {move_san}\n"
        f"Post-move FEN: {post_fen}\n\n"
        f"Verify each claim:\n"
    )
    for i, atom in enumerate(atoms, 1):
        user_msg += f"{i}. {atom}\n"
    user_msg += "\nUse tools to check facts, then output JSON verdicts."

    messages = [
        {"role": "system", "content": VERIFY_SYSTEM},
        {"role": "user", "content": user_msg},
    ]
    text, tool_log = _call_with_tools(messages, provider=provider, model=model, temperature=0.0)
    data = _parse_judge_json(text)
    assessment = data.get('assessment', 'inconclusive')
    return data.get('results', []), assessment, tool_log


# ── Step 3: Match gold atoms ──────────────────────────────────────────────

MATCH_SYSTEM = """\
Compare two sets of atomic chess claims.

Given CANDIDATE atoms (from a generated explanation) and GOLD atoms \
(from an expert annotation), determine which gold atoms are COVERED \
by the candidate atoms.

A gold atom is \"covered\" if any candidate atom expresses the same \
factual insight, even if worded differently. Partial coverage counts \
if the core insight is present.

Output valid JSON:
{"results": [{"gold_atom": "...", "covered": true, \
"matching_candidate": "the matching candidate text"}, ...]}

Use null for matching_candidate when covered is false."""


def match_gold_atoms(candidate_atoms, gold_atoms, provider=PROVIDER, model=MODEL):
    """Check which gold atoms are covered by candidate atoms."""
    if not gold_atoms:
        return []
    if not candidate_atoms:
        return [{"gold_atom": g, "covered": False, "matching_candidate": None}
                for g in gold_atoms]

    user_msg = (
        f"Candidate atoms:\n{json.dumps(candidate_atoms, indent=2)}\n\n"
        f"Gold atoms:\n{json.dumps(gold_atoms, indent=2)}"
    )
    messages = [
        {"role": "system", "content": MATCH_SYSTEM},
        {"role": "user", "content": user_msg},
    ]
    resp, _ = call_llm(messages, provider=provider, model=model, temperature=0.0)
    data = _parse_judge_json(resp.text)
    return data.get('results', [])


# ── Step 4: Quality check ─────────────────────────────────────────────────

def check_quality_assessment(gen_assessment, wp_loss):
    """Check if LLM-judged assessment matches actual move quality."""
    if wp_loss > 30:     actual_quality = 'blunder'
    elif wp_loss > 20:   actual_quality = 'mistake'
    elif wp_loss > 10:   actual_quality = 'inaccuracy'
    else:                actual_quality = 'good'

    if actual_quality in ('blunder', 'mistake'):
        quality_correct = (gen_assessment == 'bad')
    elif actual_quality == 'inaccuracy':
        quality_correct = True
    else:
        quality_correct = (gen_assessment in ('good', 'inconclusive'))

    return quality_correct, actual_quality


# ── Full judge pipeline ───────────────────────────────────────────────────

def judge_explanation(entry, explanation, gold_atoms=None,
                      provider=PROVIDER, model=MODEL):
    """Full judge pipeline: decompose -> verify -> match gold -> scores.

    Returns dict with:
        factual_precision, recall, quality_correct, gen_assessment,
        actual_quality, atoms, verification, verify_tool_log, matching,
        gold_atoms
    """
    board = chess.Board(entry['fen'])
    move_san = entry.get('move_san') or board.san(chess.Move.from_uci(entry['move_uci']))

    # 1. Decompose
    print("  Decomposing into atoms...")
    atoms = decompose_to_atoms(explanation, provider=provider, model=model)
    print(f"  -> {len(atoms)} atoms")

    # 2. Verify
    print("  Verifying atoms against position...")
    verification, gen_assessment, verify_log = verify_atoms(
        atoms, entry['fen'], move_san, entry['move_uci'],
        provider=provider, model=model)
    n_verified = sum(1 for r in verification if r.get('verified'))
    print(f"  -> {n_verified}/{len(atoms)} verified")
    print(f"  -> assessment: {gen_assessment}")

    # 3. Match gold
    if gold_atoms is None:
        gold_atoms = entry.get('extracted', {}).get('reasoning', [])
    print(f"  Matching against {len(gold_atoms)} gold atoms...")
    matching = match_gold_atoms(atoms, gold_atoms, provider=provider, model=model)
    n_covered = sum(1 for r in matching if r.get('covered'))
    print(f"  -> {n_covered}/{len(gold_atoms)} covered")

    # 4. Quality check
    wp_loss = entry.get('wp_loss', 0)
    quality_correct, actual_quality = check_quality_assessment(gen_assessment, wp_loss)
    print(f"  Quality: gen={gen_assessment}, actual={actual_quality}"
          f" -> {'correct' if quality_correct else 'WRONG'}")

    factual_precision = n_verified / len(atoms) if atoms else 0.0
    recall = n_covered / len(gold_atoms) if gold_atoms else 0.0

    return {
        'factual_precision': factual_precision,
        'recall': recall,
        'quality_correct': quality_correct,
        'gen_assessment': gen_assessment,
        'actual_quality': actual_quality,
        'atoms': atoms,
        'verification': verification,
        'verify_tool_log': verify_log,
        'matching': matching,
        'gold_atoms': gold_atoms,
    }


print('Judge pipeline ready.')

---
## Visualization Helpers

In [ ]:
def show_position(idx):
    """Display board + metadata for accepted_positions[idx]."""
    pos = accepted_positions[idx]
    board = chess.Board(pos['fen'])
    move = chess.Move.from_uci(pos['move_uci'])
    is_bad = pos.get('quality', 'good') in ('bad', 'blunder', 'mistake', 'inaccuracy')
    arrow_color = '#cc0000' if is_bad else '#15781B'

    svg = chess.svg.board(board, arrows=[(move.from_square, move.to_square)],
                          colors={'arrow': arrow_color}, size=350)
    display(SVG(svg))

    turn = 'White' if board.turn == chess.WHITE else 'Black'
    display(Markdown(
        f'### [{idx}] Position {pos["position_number"]} \u2014 '
        f'**{pos["game"]}**\n'
        f'**{turn} plays {pos["move_san"]}** '
        f'({pos.get("quality","?")} , wp_loss={pos.get("wp_loss",0):.1f}%)\n\n'
        f'`{pos["fen"]}`'
    ))

    if pos.get('engine_lines'):
        pv = []
        for i, line in enumerate(pos['engine_lines'], 1):
            tag = ' <- played' if line['move_uci'] == pos['move_uci'] else ''
            pv.append(f'{i}. **{line["move_san"]}** ({line["eval"]}): {line["pv_san"]}{tag}')
        display(Markdown('**Engine:**\n' + '\n'.join(pv)))

    gold = pos['extracted'].get('reasoning', [])
    display(Markdown('**Gold atoms:**\n' + '\n'.join(f'- {a}' for a in gold)))
    return pos


def show_board(fen, move_uci=None, size=350, bad_move=False):
    """Display a board with optional move arrow."""
    board = chess.Board(fen)
    kwargs = {'size': size}
    if move_uci:
        move = chess.Move.from_uci(move_uci)
        arrow_color = '#cc0000' if bad_move else '#15781B'
        kwargs['arrows'] = [(move.from_square, move.to_square)]
        kwargs['colors'] = {'arrow': arrow_color}
    svg = chess.svg.board(board, **kwargs)
    display(SVG(svg))


def show_judge_results(results, gold_atoms):
    """Display formatted judge results."""
    n_ver = sum(1 for r in results['verification'] if r.get('verified'))
    n_cov = sum(1 for r in results['matching'] if r.get('covered'))
    q_label = 'correct' if results['quality_correct'] else 'WRONG'
    q_detail = f"gen={results['gen_assessment']}, actual={results['actual_quality']}"

    display(Markdown(
        f'| Metric | Score |\n|--------|-------|\n'
        f'| Factual Precision | **{results["factual_precision"]:.0%}** '
        f'({n_ver}/{len(results["atoms"])}) |\n'
        f'| Recall | **{results["recall"]:.0%}** '
        f'({n_cov}/{len(gold_atoms)}) |\n'
        f'| Quality | **{q_label}** ({q_detail}) |\n'
    ))

    display(Markdown('**Generated atoms:**\n' +
        '\n'.join(f'{i+1}. {a}' for i, a in enumerate(results['atoms']))))

    display(Markdown('**Candidate atoms (verified):**'))
    for r in results['verification']:
        v = 'Y' if r.get('verified') else 'N'
        reason = f'  \n  *{r["reasoning"]}*' if r.get('reasoning') else ''
        display(Markdown(f'- [{v}] {r.get("atom", "?")}{reason}'))

    display(Markdown('**Gold atoms (covered):**'))
    for r in results['matching']:
        v = 'Y' if r.get('covered') else 'N'
        match = f'  \n  *matched: {r["matching_candidate"]}*' if r.get('matching_candidate') else ''
        display(Markdown(f'- [{v}] {r.get("gold_atom", "?")}{match}'))


def show_engine_lines(engine_lines, played_uci=None):
    """Display engine analysis lines."""
    for i, line in enumerate(engine_lines, 1):
        is_top = line.get('is_top', True)
        tag = ' <- played' if line.get('move_uci') == played_uci else ''
        prefix = f'{i}.' if is_top else '*Played:*'
        display(Markdown(f'{prefix} **{line["move_san"]}** ({line["eval"]}): {line["pv_san"]}{tag}'))


print('Visualization helpers ready.')

---
## Testing Functions

In [ ]:
def test_gen(idx, provider=PROVIDER, model=MODEL, ascii=False):
    """Generate NL commentary, display side-by-side with Chernev + gold atoms."""
    pos = accepted_positions[idx]
    show_position(idx)

    el = pos.get('engine_lines') or get_engine_analysis(pos['fen'], pos['move_uci'])
    entry = {'fen': pos['fen'], 'move_uci': pos['move_uci'],
             'annotation': pos.get('annotation', ''), 'wp_loss': pos.get('wp_loss', 0)}

    print(f"\nGenerating ({provider}/{model})...")
    text, tool_log = generate_commentary(entry, el, provider=provider, model=model, ascii=ascii)

    if tool_log:
        display(Markdown(f'*{len(tool_log)} tool call(s)*'))

    gold_atoms = pos['extracted'].get('reasoning', [])
    display(Markdown('#### Generated\n> ' + text))
    display(Markdown('#### Chernev\n> ' + pos.get('annotation', '(none)')))
    display(Markdown('#### Gold atoms\n' + '\n'.join(f'- {a}' for a in gold_atoms)))

    return text, tool_log


def test_gen_raw(idx, provider=PROVIDER, model=MODEL, ascii=False):
    """Generate commentary without engine info (tool-augmented)."""
    pos = accepted_positions[idx]
    show_position(idx)

    entry = {'fen': pos['fen'], 'move_uci': pos['move_uci'],
             'annotation': pos.get('annotation', ''), 'wp_loss': pos.get('wp_loss', 0)}

    print(f"\nGenerating RAW ({provider}/{model})...")
    text, tool_log = generate_commentary_raw(entry, provider=provider, model=model, ascii=ascii)

    if tool_log:
        display(Markdown(f'*{len(tool_log)} tool call(s)*'))

    gold_atoms = pos['extracted'].get('reasoning', [])
    display(Markdown('#### Generated (raw)\n> ' + text))
    display(Markdown('#### Chernev\n> ' + pos.get('annotation', '(none)')))
    display(Markdown('#### Gold atoms\n' + '\n'.join(f'- {a}' for a in gold_atoms)))

    return text, tool_log


def test_judge(idx, explanation, provider=PROVIDER, model=MODEL):
    """Full judge pipeline for accepted_positions[idx]."""
    pos = accepted_positions[idx]
    gold_atoms = pos['extracted'].get('reasoning', [])

    results = judge_explanation(pos, explanation, gold_atoms=gold_atoms,
                                provider=provider, model=model)

    show_judge_results(results, gold_atoms)
    return results


def test_gen_judge(idx, provider=PROVIDER, model=MODEL):
    """Generate + judge in one call."""
    text, tool_log = test_gen(idx, provider=provider, model=model)
    print("\n--- Judging ---")
    results = test_judge(idx, text, provider=provider, model=model)
    return text, results


print(f"Testing functions ready. {len(accepted_positions)} accepted positions (0-indexed).")

---
## Batch Evaluation

In [ ]:
def batch_evaluate(indices=None, n=10, seed=42, provider=PROVIDER, model=MODEL,
                   gen_model=None, output_path=None):
    """Run generation + judge on multiple positions.

    Args:
        indices: specific indices to evaluate (overrides n/seed)
        n: number of random positions
        seed: random seed
        provider: LLM provider
        model: judge model
        gen_model: generation model (defaults to model)
        output_path: optional JSONL path to save results

    Returns list of result dicts.
    """
    gen_model = gen_model or model

    if indices is None:
        rng = random.Random(seed)
        indices = rng.sample(range(len(accepted_positions)), min(n, len(accepted_positions)))

    all_results = []

    for idx in indices:
        pos = accepted_positions[idx]
        board = chess.Board(pos['fen'])
        move_san = pos.get('move_san', board.san(chess.Move.from_uci(pos['move_uci'])))
        print(f"\n{'='*60}")
        print(f"[{len(all_results)+1}/{len(indices)}] idx={idx} \u2014 {pos.get('game','?')} \u2014 {move_san}")
        print(f"{'='*60}")

        el = pos.get('engine_lines') or get_engine_analysis(pos['fen'], pos['move_uci'])
        entry = {'fen': pos['fen'], 'move_uci': pos['move_uci'],
                 'annotation': pos.get('annotation', ''), 'wp_loss': pos.get('wp_loss', 0),
                 'move_san': move_san, 'extracted': pos.get('extracted', {})}

        print(f"  Generating ({provider}/{gen_model})...")
        gen_text, gen_log = generate_commentary(entry, el, provider=provider, model=gen_model)
        print(f"  Generated: {gen_text[:100]}...")

        print(f"  Judging ({provider}/{model})...")
        gold_atoms = pos['extracted'].get('reasoning', [])
        judge_results = judge_explanation(entry, gen_text, gold_atoms=gold_atoms,
                                          provider=provider, model=model)

        result_row = {
            'idx': idx,
            'fen': pos['fen'],
            'move_san': move_san,
            'game': pos.get('game', '?'),
            'wp_loss': pos.get('wp_loss', 0),
            'quality': pos.get('quality', '?'),
            'generated_text': gen_text,
            'gen_tool_calls': len(gen_log),
            'factual_precision': judge_results['factual_precision'],
            'recall': judge_results['recall'],
            'quality_correct': judge_results['quality_correct'],
            'gen_assessment': judge_results['gen_assessment'],
            'actual_quality': judge_results['actual_quality'],
            'n_atoms': len(judge_results['atoms']),
            'n_verified': sum(1 for r in judge_results['verification'] if r.get('verified')),
            'n_gold': len(gold_atoms),
            'n_covered': sum(1 for r in judge_results['matching'] if r.get('covered')),
            'gen_model': gen_model,
            'judge_model': model,
            'provider': provider,
        }
        all_results.append(result_row)

        if output_path:
            with open(output_path, 'a') as f:
                f.write(json.dumps(result_row) + '\n')

        print(f"  Precision: {judge_results['factual_precision']:.0%} | "
              f"Recall: {judge_results['recall']:.0%} | "
              f"Quality: {'OK' if judge_results['quality_correct'] else 'WRONG'}")

    # Summary
    print(f"\n{'='*60}")
    print(f"SUMMARY ({len(all_results)} positions)")
    print(f"{'='*60}")
    avg_prec = np.mean([r['factual_precision'] for r in all_results])
    avg_rec = np.mean([r['recall'] for r in all_results])
    quality_acc = np.mean([r['quality_correct'] for r in all_results])
    print(f"  Avg Factual Precision: {avg_prec:.1%}")
    print(f"  Avg Recall: {avg_rec:.1%}")
    print(f"  Quality Accuracy: {quality_acc:.1%}")

    return all_results


def summarize_results(results):
    """Display summary table from batch_evaluate results."""
    import pandas as pd
    df = pd.DataFrame(results)
    display(Markdown('### Summary'))
    display(Markdown(
        f'| Metric | Value |\n|--------|-------|\n'
        f'| Positions | {len(df)} |\n'
        f'| Avg Precision | {df["factual_precision"].mean():.1%} |\n'
        f'| Avg Recall | {df["recall"].mean():.1%} |\n'
        f'| Quality Accuracy | {df["quality_correct"].mean():.1%} |\n'
        f'| Avg Atoms/Position | {df["n_atoms"].mean():.1f} |\n'
    ))

    # Per-quality breakdown
    if 'quality' in df.columns:
        display(Markdown('### By Move Quality'))
        grouped = df.groupby('quality').agg({
            'factual_precision': 'mean',
            'recall': 'mean',
            'quality_correct': 'mean',
            'idx': 'count',
        }).rename(columns={'idx': 'count'})
        display(grouped.style.format({
            'factual_precision': '{:.1%}',
            'recall': '{:.1%}',
            'quality_correct': '{:.1%}',
        }))

    return df


print('Batch evaluation ready.')

---
## Demo: Single Position

In [ ]:
IDX = 0  # Change this to test different positions
show_position(IDX)

In [ ]:
gen_text, gen_log = test_gen(IDX)

In [ ]:
judge_results = test_judge(IDX, gen_text)

---
## Demo: Batch Evaluation

In [ ]:
# Run batch evaluation on 5 random positions
# batch_results = batch_evaluate(n=5, seed=42)
# df = summarize_results(batch_results)

In [ ]:
# Compare models
# results_gpt = batch_evaluate(n=5, seed=42, gen_model='gpt-4o')
# results_claude = batch_evaluate(n=5, seed=42, provider='anthropic', gen_model='claude-sonnet-4-5-20250929')